# Painel — vendas 2025

Monta `saidas/painel.html` a partir dos dados tratados em `analise.ipynb`.

In [ ]:
import pandas as pd
import plotly.express as px
from analise_dados import painel

painel.tema()

In [ ]:
def reais(v):
    return f"R$ {v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

In [ ]:
df = pd.read_parquet("saidas/dados_tratados.parquet")
df.shape

## Números do painel

In [ ]:
total = df["faturamento"].sum()
reais(total)

In [ ]:
fatia_sudeste = df.loc[df["regiao"] == "Sudeste", "faturamento"].sum() / total
f"{fatia_sudeste:.0%}"

In [ ]:
margem = df["lucro"].sum() / total
f"{margem:.1%}".replace(".", ",")

## Gráficos do painel

In [ ]:
reg = df.groupby("regiao")["faturamento"].sum().sort_values().reset_index()
cores = ["#0072B2" if r == "Sudeste" else "#94A3B8" for r in reg["regiao"]]
fig_regioes = px.bar(reg, x="faturamento", y="regiao", orientation="h",
                     labels={"faturamento": "Faturamento (R$)", "regiao": "Região"})
fig_regioes.update_traces(marker_color=cores)

In [ ]:
mensal = df.set_index("data").resample("MS")["faturamento"].sum()
por_dia = (mensal / mensal.index.days_in_month).rename("por_dia").reset_index()
fig_mes = px.line(por_dia, x="data", y="por_dia", markers=True,
                  labels={"data": "Mês", "por_dia": "Faturamento por dia (R$)"})
fig_mes

## Montar o painel

In [ ]:
p = painel.Painel("Vendas 2025", "Onde está o faturamento e como ele se comportou no ano",
                  fonte="dados/vendas.csv · 5.000 vendas")
p.kpi("Faturamento em 2025", painel.compacto(total, "R$ "), f"{len(df):,} vendas".replace(",", "."), destaque=True)
p.kpi("Fatia do Sudeste", f"{fatia_sudeste:.0%}", "do faturamento, por volume")
p.kpi("Margem", f"{margem:.1%}".replace(".", ","), "igual nos três canais")

In [ ]:
p.grafico(fig_regioes, "O Sudeste lidera por volume, não por ticket",
          "O valor por venda é parecido em todas as regiões.")
p.grafico(fig_mes, "Novembro e dezembro faturam por dia mais de 60% acima de outubro",
          "Só há dados de 2025: não dá para separar fim de ano de crescimento.")
p.conclusoes(["O Sudeste concentra o faturamento porque vende mais vezes.",
              "O fim de ano pesa — falta 2024 para saber se é sazonal.",
              "A margem não muda entre canais."])
p.salvar("saidas/painel.html")

Painel salvo em `saidas/painel.html`.